In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from matplotlib.transforms import blended_transform_factory
from pathlib import Path

In [ ]:
default_dir = Path("../results_main/catalogs/default")
fg_cat = pd.read_parquet(default_dir / "galaxy_catalog_foreground.parquet")
bg_cat = pd.read_parquet(default_dir / "galaxy_catalog_background_cleaned.parquet")

fig_dir = Path("../figures")
fig_dir.mkdir(exist_ok=True)

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(7, 2), dpi=150)

# Panel 1: FlexZBoost vs LePhare photo-z
settings = dict(extent=(0, 2, 0, 2), gridsize=50, norm="log", edgecolors="none")
ax1.hexbin(
    fg_cat["fzboost_z_mode"],
    fg_cat["lephare_z_mode"],
    cmap="Greens",
    **settings,
)
ax1.hexbin(
    bg_cat["fzboost_z_mode"],
    bg_cat["lephare_z_mode"],
    cmap="Purples",
    **settings,
)
ax1.set(
    xlim=(0, 1.5),
    ylim=(0, 1.5),
    xlabel="FlexZBoost photo-z",
    ylabel="LePhare photo-z",
    aspect="equal",
)
ax1.plot([0, 2], [0, 2], c="k", lw=1, ls="--", zorder=0)

# Panel 2: Spec-z vs photo-z
ax2.hexbin(
    fg_cat["redshift"],
    fg_cat["z_phot"],
    cmap="Greens",
    **settings,
)
ax2.hexbin(
    bg_cat["redshift"],
    bg_cat["z_phot"],
    cmap="Purples",
    **settings,
)
ax2.set(
    xlim=(0, 1.5),
    ylim=(0, 1.5),
    xlabel="Spec-z",
    ylabel="Photo-z",
    aspect="equal",
)
ax2.plot([0, 2], [0, 2], c="k", lw=1, ls="--", zorder=0)

# Panel 3: Redshift histograms
# Settings for all histograms
settings = dict(range=(0, 1.5), bins=50, density=True, histtype="step")

# Foreground sample
ax3.hist(
    fg_cat["z_phot"],
    **settings,
    color="C2",
    ls="-",
)
ax3.hist(
    fg_cat["redshift"],
    **settings,
    color="C2",
    ls="--",
)

# Background sample
ax3.hist(
    bg_cat["z_phot"],
    **settings,
    color="C4",
    ls="-",
)
ax3.hist(
    bg_cat["redshift"],
    **settings,
    color="C4",
    ls="--",
)

ax3.set(
    xlim=(0, 1.5),
    # ylim=(0, 6.5),
    xlabel="Redshift",
    ylabel="Frequency",
    aspect=1.5 / 6.5,
)

ax3.hist([], histtype="step", color="k", ls="-", label="Photo-z")
ax3.hist([], histtype="step", color="k", ls="--", label="Spec-z")
ax3.legend(handlelength=1, frameon=False, fontsize=8, borderpad=0)

fig.subplots_adjust(wspace=0.4)

fig.savefig("../figures/fig_photoz_distribution.pdf", dpi=500, bbox_inches="tight")

In [ ]:
fg_red = fg_cat.query("(g_absmag - r_absmag) > 0.5")
fg_blue = fg_cat.query("(g_absmag - r_absmag) < 0.5")

fig, axes = plt.subplots(1, 2, figsize=(4, 1.75), dpi=200, sharey=True)

# First apparent magnitude
# ------------------------
axes[0].set(xlabel="Apparent $r$-band magnitude", ylabel="Number of galaxies")
axes[0].hist(fg_blue["r_cModelMag"], bins=25, alpha=0.5, color="C0")
axes[0].hist(fg_red["r_cModelMag"], bins=25, alpha=0.5, color="C3")
axes[0].set(xticks=np.arange(16, 26, 2), ylim=(0, 1e3))

# Vertical bars to indicate comparisons to other surveys
blended = blended_transform_factory(axes[0].transData, axes[0].transAxes)

axes[0].axvline(21, ls="--", c="gray")
axes[0].annotate(
    "",
    xy=(19.2, 0.94),
    xytext=(21, 0.94),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[0].text(
    21.1,
    0.94,
    "Menard",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="left",
    va="center",
)

axes[0].axvline(20, ls="--", c="gray")
axes[0].annotate(
    "",
    xy=(18.5, 0.81),
    xytext=(20, 0.81),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[0].text(
    19.8,
    0.84,
    "Genc",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="right",
    va="bottom",
)

axes[0].axvline(17.77, ls="--", c="gray")
axes[0].annotate(
    "",
    xy=(17.77 - 1.5, 0.81),
    xytext=(17.77, 0.81),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[0].text(
    17.57,
    0.84,
    "Peek",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="right",
    va="bottom",
)

# Now stellar mass
# ------------------------
axes[1].set(xlabel="Stellar mass [$M_\\odot$]", xscale="log")
axes[1].hist(
    fg_blue["stellar_mass"],
    bins=np.geomspace(0.9e8, 4e11, 25),
    alpha=0.5,
    color="C0",
)
axes[1].hist(
    fg_red["stellar_mass"],
    bins=np.geomspace(0.9e8, 4e11, 25),
    alpha=0.5,
    color="C3",
)

# Vertical bars to indicate comparisons to other surveys
blended = blended_transform_factory(axes[1].transData, axes[1].transAxes)

axes[1].axvline(10**9, ls="--", c="gray")
axes[1].annotate(
    "",
    xy=(10**10, 0.81),
    xytext=(10**9, 0.81),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[1].text(
    10**9.1,
    0.84,
    "Peek",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="left",
    va="bottom",
)

axes[1].axvline(10**10.2, ls="--", c="gray")
axes[1].annotate(
    "",
    xy=(10**11.2, 0.81),
    xytext=(10**10.2, 0.81),
    xycoords=blended,
    textcoords=blended,
    arrowprops=dict(arrowstyle="->", color="gray", lw=0.8),
)
axes[1].text(
    10**10.3,
    0.84,
    "Menard,\nGenc",
    transform=blended,
    color="gray",
    fontsize=7,
    ha="left",
    va="bottom",
)

fig.savefig("../figures/fig_foreground_properties.pdf", bbox_inches="tight")

In [ ]:
joint = pd.concat([fg_blue, fg_red])
joint["set"] = ["blue"] * len(fg_blue) + ["red"] * len(fg_red)

jp = sns.jointplot(
    x=joint["r_absmag"],
    y=joint["g_absmag"] - joint["r_absmag"],
    hue=joint["set"],
    palette=["C0", "C3"],
    kind="kde",
    legend=False,
    levels=8,
    joint_kws={"thresh": 0.1},
    xlim=(-22.5, -16.75),
    ylim=(0.1, 0.8),
)

fig = jp.figure
fig.set_figwidth(2)
fig.set_figheight(1.75)
fig.set_dpi(200)

# Set axis labels
jp.set_axis_labels("Absolute $r$-band magnitude", "Rest-frame $g - r$ color")

jp.savefig("../figures/fig_foreground_cmd.pdf", bbox_inches="tight")